# 00 — Solution Feasibility Diagnostics (run BEFORE choosing a product)
**Consulting rule:** kill bad ideas cheaply. Each check below either *justifies* or *kills*
a candidate AI product before we invest days in it.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

df = pd.read_csv("olist_master_clean.csv")
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])

print("=" * 55)
print("GATE 1: GRAIN AUDIT — is one row really one order?")
print("=" * 55)
dupes = df["order_id"].duplicated().sum()
print(f"rows: {len(df):,} | unique orders: {df['order_id'].nunique():,} | duplicates: {dupes}")
assert dupes == 0, "ABORT: duplicate orders — fix grain before anything else!"
print("PASS ✅\n")

print("=" * 55)
print("GATE 2: TEMPORAL COVERAGE — how long is our time axis?")
print("=" * 55)
t = df["order_purchase_timestamp"]
print(f"first order: {t.min()}  |  last order: {t.max()}")
print(f"span: {(t.max() - t.min()).days / 30.4:.1f} months")
print("-> long enough for time-based split AND survival analysis ✅\n")

print("=" * 55)
print("GATE 3+4: TARGET CONSTRUCTIBILITY & BASE RATE")
print("=" * 55)
df["is_late"] = (df["delivery_delay_days"] > 0).astype(int)
rate = df["is_late"].mean()
print(f"Target 'is_late' (delay > 0 days): {rate:.2%} of orders")
print(f"Target 'very_late' (> 3 days):     {(df['delivery_delay_days'] > 3).mean():.2%} of orders")
print("-> both constructible; ~8% base rate is workable for classification\n")

print("=" * 55)
print("GATE 5: SIGNAL AUDIT — do raw features know anything?")
print("=" * 55)
df["estimated_window_days"] = (
    pd.to_datetime(df["order_estimated_delivery_date"]) - t
).dt.days
df["freight_price_ratio"] = df["freight_value"] / (df["price"] + 1)

for feat in ["estimated_window_days", "freight_value", "freight_price_ratio", "price", "n_items"]:
    auc = roc_auc_score(df["is_late"], df[feat])
    verdict = "signal ✅" if auc > 0.56 else ("weak ⚠️" if auc > 0.52 else "none ❌")
    print(f"  {feat:25s} univariate AUC = {auc:.3f}  {verdict}")
print("(AUC 0.5 = feature knows nothing; >0.6 = real signal worth building on)\n")

print("=" * 55)
print("GATE 6: DEPTH-PER-ENTITY — can we model at SELLER level?")
print("=" * 55)
per_seller = df.groupby("seller_id")["order_id"].nunique()
print(f"sellers: {len(per_seller):,}")
print(f"orders per seller — median: {per_seller.median():.0f} | p90: {per_seller.quantile(0.9):.0f} | max: {per_seller.max():,}")
print("-> enough history per seller for a SELLER-LEVEL model ✅\n")

print("=" * 55)
print("BONUS CHECK: is SELLER CHURN constructible? (for the survival model)")
print("=" * 55)
LAST_DAY = t.max()
seller_last = df.groupby("seller_id")["order_purchase_timestamp"].max()
seller_first = df.groupby("seller_id")["order_purchase_timestamp"].min()
days_inactive = (LAST_DAY - seller_last).dt.days
churned = days_inactive > 90          # no order in last 3 months = churned
censored = ~churned                   # still active at dataset end = censored
print(f"sellers inactive >90 days at dataset end: {churned.sum():,} ({churned.mean():.1%})  <- events")
print(f"sellers still active (censored):          {censored.sum():,} ({censored.mean():.1%})  <- censored")
print("-> event + censoring both exist -> SURVIVAL ANALYSIS is viable 🔥")
print(f"median seller lifespan (first->last order): {(seller_last - seller_first).dt.days.median():.0f} days")
